# Tire Anomaly Detection — Model Training & Evaluation

This notebook trains a SageMaker Random Cut Forest model for tire pressure anomaly detection.

**What it does:**
1. Loads the synthetic training dataset (721K records, 50 vehicles, 6 months)
2. Prepares and normalizes features (pressure, temperature, delta_pressure, delta_temp)
3. Trains an RCF model on normal data only (unsupervised anomaly detection)
4. Evaluates on labeled test data (slow leaks, punctures, valve failures)
5. Deploys a real-time inference endpoint

**Prerequisites:**
- Run `python3 scripts/generate_training_data.py` first to create the dataset
- SageMaker execution role with S3 and SSM access
- S3 bucket for training artifacts

In [ ]:
import boto3
import json
import io
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# Configuration
REGION = 'us-east-2'
BUCKET = 'cms-tire-prediction-ACCOUNT-REGION'  # Update with your bucket
ROLE_ARN = 'arn:aws:iam::ACCOUNT:role/cms-sagemaker-execution-role'  # Update
STAGE = 'prod'

## 1. Load and Explore Training Data

In [ ]:
df = pd.read_parquet('../data/training/tire_telemetry_full.parquet')
print(f'Dataset: {len(df):,} records')
print(f'Vehicles: {df.vehicle_id.nunique()}')
print(f'Date range: {df.timestamp.min()} → {df.timestamp.max()}')
print(f'\nLabel distribution:')
print(df.label.value_counts())

In [ ]:
# Visualize pressure distribution by label
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label in df.label.unique():
    subset = df[df.label == label]
    axes[0].hist(subset.pressure, bins=50, alpha=0.5, label=label)
axes[0].set_xlabel('Pressure (PSI)')
axes[0].set_ylabel('Count')
axes[0].set_title('Pressure Distribution by Label')
axes[0].legend()
axes[0].axvline(x=28, color='red', linestyle='--', label='Alert threshold')

# Show a slow leak example
leak = df[(df.label == 'slow_leak') & (df.tire_id == 'FL')].sort_values('timestamp').head(500)
if len(leak) > 0:
    vid = leak.vehicle_id.iloc[0]
    vehicle_leak = df[(df.vehicle_id == vid) & (df.tire_id == 'FL')].sort_values('timestamp')
    axes[1].plot(range(len(vehicle_leak)), vehicle_leak.pressure.values, linewidth=0.5)
    axes[1].axhline(y=28, color='red', linestyle='--', label='Alert threshold')
    axes[1].set_xlabel('Reading #')
    axes[1].set_ylabel('Pressure (PSI)')
    axes[1].set_title(f'Slow Leak Example ({vid} FL)')
    axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Prepare Features

In [ ]:
features = ['pressure', 'temperature', 'delta_pressure', 'delta_temp']

# Train on normal data only (unsupervised)
normal = df[df.label == 'normal'][features].dropna()
test = df[features + ['label']].dropna()

# Normalize
stats = {}
for col in features:
    stats[col] = {'mean': float(normal[col].mean()), 'std': float(normal[col].std())}

train_norm = normal.copy()
test_norm = test.copy()
for col in features:
    train_norm[col] = (train_norm[col] - stats[col]['mean']) / stats[col]['std']
    test_norm[col] = (test_norm[col] - stats[col]['mean']) / stats[col]['std']

print(f'Training: {len(train_norm):,} (normal only)')
print(f'Test: {len(test_norm):,} (all labels)')
print(f'\nNormalization stats:')
for k, v in stats.items():
    print(f'  {k}: mean={v["mean"]:.3f}, std={v["std"]:.3f}')

## 3. Train Random Cut Forest Model

In [ ]:
sm = boto3.client('sagemaker', region_name=REGION)
s3 = boto3.client('s3', region_name=REGION)

train_array = train_norm[features].values.astype('float32')
job_name = f'tire-rcf-{datetime.now().strftime("%Y%m%d-%H%M%S")}'
prefix = f'tire-prediction/training/{job_name}'

# Upload training CSV
buf = io.StringIO()
pd.DataFrame(train_array).to_csv(buf, header=False, index=False)
s3.put_object(Bucket=BUCKET, Key=f'{prefix}/train/train.csv', Body=buf.getvalue())
print(f'Uploaded {len(train_array):,} training samples to s3://{BUCKET}/{prefix}/train/')

# RCF container
acct_map = {'us-east-1': '382416733822', 'us-east-2': '404615174143', 'us-west-2': '174872318107'}
image = f'{acct_map.get(REGION, "404615174143")}.dkr.ecr.{REGION}.amazonaws.com/randomcutforest:latest'

sm.create_training_job(
    TrainingJobName=job_name,
    AlgorithmSpecification={'TrainingImage': image, 'TrainingInputMode': 'File'},
    RoleArn=ROLE_ARN,
    InputDataConfig=[{'ChannelName': 'train', 'DataSource': {'S3DataSource': {
        'S3DataType': 'S3Prefix', 'S3Uri': f's3://{BUCKET}/{prefix}/train',
        'S3DataDistributionType': 'ShardedByS3Key'}}, 'ContentType': 'text/csv;label_size=0'}],
    OutputDataConfig={'S3OutputPath': f's3://{BUCKET}/{prefix}/output'},
    ResourceConfig={'InstanceType': 'ml.m5.large', 'InstanceCount': 1, 'VolumeSizeInGB': 10},
    StoppingCondition={'MaxRuntimeInSeconds': 600},
    HyperParameters={'num_samples_per_tree': '256', 'num_trees': '100', 'feature_dim': '4'},
)
print(f'Training job started: {job_name}')

# Wait
while True:
    status = sm.describe_training_job(TrainingJobName=job_name)['TrainingJobStatus']
    print(f'  {status}')
    if status in ('Completed', 'Failed', 'Stopped'): break
    time.sleep(30)

model_data = sm.describe_training_job(TrainingJobName=job_name)['ModelArtifacts']['S3ModelArtifacts']
print(f'\n✅ Model: {model_data}')

## 4. Deploy Endpoint

In [ ]:
ts = datetime.now().strftime('%Y%m%d-%H%M%S')
endpoint_name = f'tire-anomaly-{datetime.now().strftime("%Y%m%d")}'
model_name = f'tire-rcf-{ts}'
config_name = f'tire-rcf-cfg-{ts}'

sm.create_model(ModelName=model_name, ExecutionRoleArn=ROLE_ARN,
                PrimaryContainer={'Image': image, 'ModelDataUrl': model_data})

sm.create_endpoint_config(EndpointConfigName=config_name, ProductionVariants=[{
    'VariantName': 'default', 'ModelName': model_name,
    'InstanceType': 'ml.m5.large', 'InitialInstanceCount': 1}])

sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=config_name)
print(f'Creating endpoint: {endpoint_name}')

while True:
    status = sm.describe_endpoint(EndpointName=endpoint_name)['EndpointStatus']
    print(f'  {status}')
    if status in ('InService', 'Failed'): break
    time.sleep(30)

print(f'\n✅ Endpoint ready: {endpoint_name}')

## 5. Evaluate Model

In [ ]:
sm_runtime = boto3.client('sagemaker-runtime', region_name=REGION)

test_array = test_norm[features].values.astype('float32')
labels = test_norm['label'].values

# Get anomaly scores in batches
scores = []
batch_size = 500
for i in range(0, min(len(test_array), 10000), batch_size):  # Sample 10K for speed
    batch = test_array[i:i+batch_size]
    body = '\n'.join(','.join(str(v) for v in row) for row in batch)
    resp = sm_runtime.invoke_endpoint(
        EndpointName=endpoint_name, ContentType='text/csv', Body=body)
    result = json.loads(resp['Body'].read().decode())
    scores.extend([r['score'] for r in result['scores']])
    if i % 2000 == 0: print(f'  {i}/{min(len(test_array), 10000)}')

scores = np.array(scores)
sample_labels = labels[:len(scores)]

# Threshold at 95th percentile of normal scores
normal_scores = scores[sample_labels == 'normal']
anomaly_scores = scores[sample_labels != 'normal']
threshold = float(np.percentile(normal_scores, 95))

print(f'Threshold: {threshold:.4f}')
print(f'Normal scores:  mean={normal_scores.mean():.4f}, p95={np.percentile(normal_scores, 95):.4f}')
print(f'Anomaly scores: mean={anomaly_scores.mean():.4f}, p95={np.percentile(anomaly_scores, 95):.4f}')

# Metrics
predictions = scores > threshold
true_anomalies = sample_labels != 'normal'
tp = np.sum(predictions & true_anomalies)
fp = np.sum(predictions & ~true_anomalies)
fn = np.sum(~predictions & true_anomalies)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f'\nPrecision: {precision:.3f}')
print(f'Recall:    {recall:.3f}')
print(f'F1 Score:  {f1:.3f}')

In [ ]:
# Visualize score distributions
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(normal_scores, bins=50, alpha=0.5, label='Normal', color='green')
ax.hist(anomaly_scores, bins=50, alpha=0.5, label='Anomaly', color='red')
ax.axvline(x=threshold, color='black', linestyle='--', label=f'Threshold ({threshold:.2f})')
ax.set_xlabel('Anomaly Score')
ax.set_ylabel('Count')
ax.set_title('RCF Anomaly Score Distribution')
ax.legend()
plt.show()

## 6. Save Configuration to SSM

In [ ]:
ssm = boto3.client('ssm', region_name=REGION)
prefix = f'/tire-prediction/{STAGE}'

ssm.put_parameter(Name=f'{prefix}/normalization-stats', Value=json.dumps(stats), Type='String', Overwrite=True)
ssm.put_parameter(Name=f'{prefix}/anomaly-threshold', Value=json.dumps({'threshold': threshold}), Type='String', Overwrite=True)
ssm.put_parameter(Name=f'{prefix}/endpoint-name', Value=endpoint_name, Type='String', Overwrite=True)

print(f'✅ Config saved to SSM ({prefix}/*)')
print(f'  Normalization stats: {json.dumps(stats, indent=2)}')
print(f'  Threshold: {threshold}')
print(f'  Endpoint: {endpoint_name}')